md# 02 · Fog co-occurrence figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sladekd/fog-obs-fusion/blob/main/notebooks/02_fog_figures.ipynb)

Four figures in one dark style: the fog web map, fog regions, the fog clock, and an animation of the foggiest night. Shared fog between two stations is measured with the Jaccard index (hours both in fog / hours either in fog).

In [ ]:
# Install the package from GitHub (Colab). Locally: pip install -e .
!pip install -q git+https://github.com/Sladekd/fog-obs-fusion.git

In [ ]:
import pandas as pd

# Load your decoded SYNOP table (columns WMO_ID, ANO, MES, DIA, HORA, visibility_m, ...).
# In Colab, e.g. mount Drive first:
#   from google.colab import drive; drive.mount("/content/drive")
final_df = pd.read_parquet("/content/drive/MyDrive/synop/final_df.parquet")   # <- edit path
final_df.head()

In [ ]:
from fogfusion.io import load_stations, check_coverage, wide, FOG_VIS_M
from fogfusion.graph import build_delaunay_graph, jaccard_matrix
from fogfusion.viz import fog_web, fog_regions, fog_clock, fog_night_gif
from fogfusion.viz.style import BG
import matplotlib.pyplot as plt

sg = build_delaunay_graph(check_coverage(final_df, load_stations()))
vis = wide(final_df, "visibility_m", agg="min")
vis = vis[[c for c in vis.columns if c in set(sg.stations.WMO_ID)]]
fog = (vis < FOG_VIS_M).astype(float).where(vis.notna())
J = jaccard_matrix(fog)
print((100 * fog.mean()).round(1).sort_values(ascending=False).head(10))

md## Fog web

In [ ]:
fig = fog_web(fog, J, sg.stations, sg.edges)
fig.savefig("fog_web.png", dpi=200, facecolor=BG); plt.show()

md## Fog regions

In [ ]:
fig = fog_regions(J, sg.stations)
fig.savefig("fog_regions.png", dpi=200, facecolor=BG); plt.show()

md## Fog clock

In [ ]:
fig = fog_clock(fog)
fig.savefig("fog_clock.png", dpi=200, facecolor=BG); plt.show()

md## The foggiest night (GIF)

In [ ]:
from IPython.display import Image
fog_night_gif(vis, sg.stations, sg.edges, "fog_night.gif")
Image(filename="fog_night.gif")